# Visualisasi Bab 3 — Hasil & Pembahasan
**Skripsi:** Klasifikasi Komentar Publik Layanan Digital Pemerintah  
**Sumber data:** Dibaca langsung dari file aktual di folder `xgboost_method`  

**Urutan cell:**
1. Setup & Load Data
2. Gambar 7 — Distribusi Kelas (Labeling)
3. Gambar 9 — Distribusi Kelas (Preprocessing)
4. Gambar 10 — Data Splitting
5. Gambar 11 — Top 10 TF-IDF
6. Gambar 12 — Top 10 Chi-Square
7. Gambar 13–16 — Confusion Matrix (E-01, E-06, E-07, E-12)
8. Gambar 17 — Perbandingan Akurasi & Macro F1
9. Gambar 18 — F1-Score Per Kelas

## Cell 1 — Setup & Load Data

In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import scipy.sparse
import pickle
import os

# ── Path folder project ──────────────────────────────────────────────────────
# Sesuaikan dengan lokasi folder project Anda
BASE = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_method'
OUT  = 'output_bab3'
os.makedirs(OUT, exist_ok=True)

# ── Style global (Arial, Draw.io-like) ───────────────────────────────────────
plt.rcParams.update({
    'font.family'        : 'Arial',
    'font.size'          : 10,
    'axes.titlesize'     : 11,
    'axes.titleweight'   : 'bold',
    'axes.labelsize'     : 10,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'axes.grid'          : True,
    'axes.grid.axis'     : 'y',
    'grid.alpha'         : 0.3,
    'grid.linestyle'     : '--',
    'figure.dpi'         : 200,
    'savefig.dpi'        : 200,
    'savefig.bbox'       : 'tight',
    'savefig.facecolor'  : 'white',
    'savefig.pad_inches' : 0.15,
})

# Warna konsisten
CLR = {'keluhan':'#C0392B','saran':'#D4890A','pujian':'#1A7A4A',
       'lr':'#7F8C8D','xgb':'#2471A3','bert':'#6C3483','line':'#D4890A'}

def save_fig(fig, name):
    path = f'{OUT}/{name}'
    fig.savefig(path)
    plt.close(fig)
    print(f'  ✅ {name}')

# ── Load data aktual ─────────────────────────────────────────────────────────
print('Memuat data...')

# File CSV utama
df_labelled = pd.read_csv(os.path.join(BASE, 'labelled_data_final.csv'))
df_prep     = pd.read_csv(os.path.join(BASE, 'data_preprocessing_3.0.csv'))
df_train    = pd.read_csv(os.path.join(BASE, 'data_train_final_3.0.csv'))
df_test     = pd.read_csv(os.path.join(BASE, 'data_test_final_3.0.csv'))

# Sparse matrix TF-IDF + Chi-Square
X_train = scipy.sparse.load_npz(os.path.join(BASE, 'X_train_selected_3.0.npz'))
X_test  = scipy.sparse.load_npz(os.path.join(BASE, 'X_test_selected_3.0.npz'))

# Label mapping
LABEL_MAP = {'keluhan':0,'saran':1,'pujian':2}
INV_MAP   = {0:'keluhan',1:'saran',2:'pujian'}

# Encode label
for df in [df_labelled, df_prep, df_train, df_test]:
    if 'label_pks' in df.columns and 'label_enc' not in df.columns:
        df['label_enc'] = df['label_pks'].map(LABEL_MAP)

y_train    = df_train['label_enc'].values
y_test     = df_test['label_enc'].values
y_test_str = df_test['label_pks'].values

print(f'df_labelled : {len(df_labelled):,} baris')
print(f'df_prep     : {len(df_prep):,} baris')
print(f'df_train    : {len(df_train):,} baris')
print(f'df_test     : {len(df_test):,} baris')
print(f'X_train     : {X_train.shape}')
print(f'X_test      : {X_test.shape}')
print('\nSemua data berhasil dimuat!')

Memuat data...
df_labelled : 80,163 baris
df_prep     : 70,135 baris
df_train    : 56,108 baris
df_test     : 14,027 baris
X_train     : (56108, 3000)
X_test      : (14027, 3000)

Semua data berhasil dimuat!


## Gambar 7 — Distribusi Kelas Data Labeling

In [3]:
# Hitung distribusi dari data aktual
vc = df_labelled['label_pks'].value_counts()
labels_order = ['keluhan','saran','pujian']
values = [vc.get(l,0) for l in labels_order]
total  = sum(values)

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(labels_order, values,
              color=[CLR[l] for l in labels_order],
              width=0.48, edgecolor='white', linewidth=0.8, zorder=3)

for bar, val in zip(bars, values):
    pct = val/total*100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
            f'{val:,}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=9.5, fontweight='bold',
            color=bar.get_facecolor())

ax.set_title(f'Distribusi Kelas — Data Labeling (Total: {total:,})', pad=10)
ax.set_ylabel('Jumlah Komentar')
ax.set_ylim(0, max(values)*1.22)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.set_xlabel('Kategori Label')

fig.tight_layout()
save_fig(fig, 'Gambar07_Distribusi_Labeling.png')
plt.show()

  ✅ Gambar07_Distribusi_Labeling.png


## Gambar 9 — Distribusi Kelas Setelah Preprocessing

In [4]:
vc2 = df_prep['label_pks'].value_counts()
values2 = [vc2.get(l,0) for l in labels_order]
total2  = sum(values2)

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(labels_order, values2,
              color=[CLR[l] for l in labels_order],
              width=0.48, edgecolor='white', linewidth=0.8, zorder=3)

for bar, val in zip(bars, values2):
    pct = val/total2*100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+250,
            f'{val:,}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=9.5, fontweight='bold',
            color=bar.get_facecolor())

ax.set_title(f'Distribusi Kelas — Setelah Preprocessing (Total: {total2:,})', pad=10)
ax.set_ylabel('Jumlah Komentar')
ax.set_ylim(0, max(values2)*1.22)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.set_xlabel('Kategori Label')

fig.tight_layout()
save_fig(fig, 'Gambar09_Distribusi_Preprocessing.png')
plt.show()

  ✅ Gambar09_Distribusi_Preprocessing.png


## Gambar 10 — Data Splitting (Train vs Test per Kelas)

In [5]:
vc_tr = df_train['label_pks'].value_counts()
vc_te = df_test['label_pks'].value_counts()
tr_vals = [vc_tr.get(l,0) for l in labels_order]
te_vals = [vc_te.get(l,0) for l in labels_order]

x = np.arange(len(labels_order))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x-w/2, tr_vals, w, label=f'Training (80%) — {len(df_train):,}',
            color='#2471A3', edgecolor='white', zorder=3)
b2 = ax.bar(x+w/2, te_vals, w, label=f'Testing  (20%) — {len(df_test):,}',
            color='#E67E22', edgecolor='white', zorder=3)

for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+120,
            f'{int(bar.get_height()):,}',
            ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_title('Perbandingan Distribusi Data Training dan Testing', pad=10)
ax.set_ylabel('Jumlah Sampel')
ax.set_xticks(x); ax.set_xticklabels(labels_order)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend(fontsize=9)

fig.tight_layout()
save_fig(fig, 'Gambar10_Data_Splitting.png')
plt.show()

  ✅ Gambar10_Data_Splitting.png


## Gambar 11–12 — Load TF-IDF & Chi-Square
Gambar 11 dan 12 dibuat dari objek TF-IDF dan selector Chi-Square yang tersimpan dari eksperimen E-07.

In [7]:
# Load TF-IDF vectorizer dan Chi-Square selector yang disimpan saat eksperimen
# Sesuaikan nama file dengan yang ada di folder project Anda
TFIDF_FILE    = os.path.join(BASE, 'tfidf_model_3.0.pkl')   # ganti nama sesuai file
SELECTOR_FILE = os.path.join(BASE, 'selector_chi2_3.0.pkl')          # ganti nama sesuai file
MODEL_E07     = os.path.join(BASE, 'model_xgboost_tuned_3.0.pkl')

with open(TFIDF_FILE,    'rb') as f: tfidf    = pickle.load(f)
with open(SELECTOR_FILE, 'rb') as f: selector = pickle.load(f)
with open(MODEL_E07,     'rb') as f: model_e07= pickle.load(f)

# Rekonstruksi nama fitur terpilih (Chi-Square)
all_feats      = tfidf.get_feature_names_out()
selected_mask  = selector.get_support()
selected_feats = all_feats[selected_mask]

# Score Chi-Square
chi2_scores   = selector.scores_[selected_mask]
tfidf_weights = np.asarray(X_train.mean(axis=0)).flatten()

print(f'Total fitur TF-IDF        : {len(all_feats):,}')
print(f'Fitur terpilih Chi-Square : {len(selected_feats):,}')
print('Siap buat Gambar 11 dan 12')

Total fitur TF-IDF        : 39,645
Fitur terpilih Chi-Square : 3,000
Siap buat Gambar 11 dan 12


## Gambar 11 — Top 10 Feature Extraction (TF-IDF)

In [8]:
top_n    = 10
top_idx  = np.argsort(tfidf_weights)[::-1][:top_n]
top_words= selected_feats[top_idx]
top_vals = tfidf_weights[top_idx]

fig, ax = plt.subplots(figsize=(8, 5))
colors  = ['#1A4F7A' if v > np.median(top_vals) else '#5B8DB8' for v in top_vals]
bars = ax.barh(range(top_n), top_vals, color=colors,
               edgecolor='white', linewidth=0.8, zorder=3)

for bar, val, word in zip(bars, top_vals, top_words):
    ax.text(bar.get_width()+0.0003, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8.5)

ax.set_yticks(range(top_n))
ax.set_yticklabels(top_words, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Rata-rata Bobot TF-IDF')
ax.set_title('Top 10 Feature Extraction (TF-IDF)', pad=10)
ax.axvline(np.mean(top_vals), color='#D4890A', linestyle='--',
           linewidth=1.2, label='Rata-rata', alpha=0.8)
ax.legend(fontsize=9)

fig.tight_layout()
save_fig(fig, 'Gambar11_Feature_Extraction_TFIDF.png')
plt.show()

  ✅ Gambar11_Feature_Extraction_TFIDF.png


## Gambar 12 — Top 10 Feature Selection (Chi-Square)

In [9]:
top_chi_idx   = np.argsort(chi2_scores)[::-1][:top_n]
top_chi_words = selected_feats[top_chi_idx]
top_chi_vals  = chi2_scores[top_chi_idx]

fig, ax = plt.subplots(figsize=(8, 5))
colors2 = ['#7B1D0A' if v > np.median(top_chi_vals) else '#C0392B' for v in top_chi_vals]
bars = ax.barh(range(top_n), top_chi_vals, color=colors2,
               edgecolor='white', linewidth=0.8, zorder=3)

for bar, val in zip(bars, top_chi_vals):
    ax.text(bar.get_width()+1.5, bar.get_y()+bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=8.5)

ax.set_yticks(range(top_n))
ax.set_yticklabels(top_chi_words, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Chi-Square Score')
ax.set_title('Top 10 Feature Selection (Chi-Square, k=1.500)', pad=10)

fig.tight_layout()
save_fig(fig, 'Gambar12_Feature_Selection_ChiSquare.png')
plt.show()

  ✅ Gambar12_Feature_Selection_ChiSquare.png


## Load Model E-01, E-06, E-07, E-12 untuk Confusion Matrix
Sesuaikan path file model dengan yang ada di folder project Anda.

In [ ]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    classification_report, f1_score
)

# ── E-01: LR Baseline (dataset kecil, 14.982) ────────────────────────────────
# Load data kecil E-01
df_train_e01 = pd.read_csv(os.path.join(BASE, 'data_train_final_1.0.csv'))  # sesuaikan nama
df_test_e01  = pd.read_csv(os.path.join(BASE, 'data_test_final_1.0.csv'))   # sesuaikan nama
X_train_e01  = scipy.sparse.load_npz(os.path.join(BASE, 'X_train_selected_v1.npz'))  # sesuaikan
X_test_e01   = scipy.sparse.load_npz(os.path.join(BASE, 'X_test_selected_v1.npz'))   # sesuaikan
y_test_e01   = df_test_e01['label_pks'].values

MODEL_E01 = os.path.join(BASE, 'model_lr_baseline.pkl')          # sesuaikan nama
MODEL_E06 = os.path.join(BASE, 'model_lr_big.pkl')               # sesuaikan nama
MODEL_E12 = os.path.join(BASE, '..', 'indobert_finetuned_v2',    # sesuaikan path
                         'predictions_e12.csv')                   # CSV prediksi E-12

with open(MODEL_E01,'rb') as f: model_e01 = pickle.load(f)
with open(MODEL_E06,'rb') as f: model_e06 = pickle.load(f)

# Prediksi masing-masing model
y_pred_e01 = [INV_MAP[p] for p in model_e01.predict(X_test_e01)]
y_pred_e06 = [INV_MAP[p] for p in model_e06.predict(X_test)]
y_pred_e07 = [INV_MAP[p] for p in model_e07.predict(X_test)]

# E-12 IndoBERT: load dari file CSV hasil prediksi (format: kolom 'y_true','y_pred')
df_e12     = pd.read_csv(MODEL_E12)
y_true_e12 = df_e12['y_true'].values
y_pred_e12 = df_e12['y_pred'].values

print('Akurasi masing-masing model:')
print(f'  E-01 LR baseline : {accuracy_score(y_test_e01, y_pred_e01)*100:.2f}%')
print(f'  E-06 LR big      : {accuracy_score(y_test_str, y_pred_e06)*100:.2f}%')
print(f'  E-07 XGBoost     : {accuracy_score(y_test_str, y_pred_e07)*100:.2f}%')
print(f'  E-12 IndoBERT v2 : {accuracy_score(y_true_e12, y_pred_e12)*100:.2f}%')

## Fungsi Confusion Matrix (dipakai Gambar 13–16)

In [ ]:
def plot_cm(y_true, y_pred, title, fname, acc_label):
    order = ['keluhan','saran','pujian']
    cm    = confusion_matrix(y_true, y_pred, labels=order)

    fig, ax = plt.subplots(figsize=(6, 5))
    vmax = cm.max()
    im   = ax.imshow(cm, cmap='Blues', vmin=0, vmax=vmax, aspect='auto')

    for i in range(3):
        for j in range(3):
            color = 'white' if cm[i,j] > vmax*0.55 else '#1A1A1A'
            ax.text(j, i, f'{cm[i,j]:,}',
                    ha='center', va='center',
                    fontsize=12, fontweight='bold', color=color)

    ax.set_xticks([0,1,2]); ax.set_xticklabels(order, fontsize=10)
    ax.set_yticks([0,1,2]); ax.set_yticklabels(order, fontsize=10)
    ax.set_xlabel('Predicted Label', fontweight='bold', labelpad=8)
    ax.set_ylabel('Actual Label',    fontweight='bold', labelpad=8)
    ax.set_title(f'{title}\nAccuracy: {acc_label}', pad=10)

    plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
    fig.tight_layout()
    save_fig(fig, fname)
    plt.show()

print('Fungsi plot_cm siap.')

## Gambar 13 — Confusion Matrix — LR Baseline (E-01)

In [ ]:
plot_cm(y_test_e01, y_pred_e01,
        'Confusion Matrix — LR Baseline (E-01)',
        'Gambar13_CM_E01_LR.png',
        '78.00%')

## Gambar 14 — Confusion Matrix — LR Data Besar (E-06)

In [ ]:
plot_cm(y_test_str, y_pred_e06,
        'Confusion Matrix — LR Data Besar (E-06)',
        'Gambar14_CM_E06_LR.png',
        '74.38%')

## Gambar 15 — Confusion Matrix — XGBoost Tuned v3.0 (E-07)

In [ ]:
plot_cm(y_test_str, y_pred_e07,
        'Confusion Matrix — XGBoost Tuned v3.0 (E-07)',
        'Gambar15_CM_E07_XGBoost.png',
        '80.68%')

## Gambar 16 — Confusion Matrix — IndoBERT v2 (E-12)

In [ ]:
plot_cm(y_true_e12, y_pred_e12,
        'Confusion Matrix — IndoBERT v2 (E-12)',
        'Gambar16_CM_E12_IndoBERT.png',
        '92.78%')

## Gambar 17 — Perbandingan Akurasi & Macro F1 Keempat Eksperimen

In [ ]:
# Hitung ulang dari prediksi aktual
results = [
    ('E-01\nLR Baseline\n(14.982)',
     accuracy_score(y_test_e01, y_pred_e01)*100,
     f1_score(y_test_e01, y_pred_e01, average='macro'),
     CLR['lr']),
    ('E-06\nLR Data Besar\n(70.135)',
     accuracy_score(y_test_str, y_pred_e06)*100,
     f1_score(y_test_str, y_pred_e06, average='macro'),
     CLR['lr']),
    ('E-07\nXGBoost Tuned\n(70.135)',
     accuracy_score(y_test_str, y_pred_e07)*100,
     f1_score(y_test_str, y_pred_e07, average='macro'),
     CLR['xgb']),
    ('E-12\nIndoBERT v2\n(80.163)',
     accuracy_score(y_true_e12, y_pred_e12)*100,
     f1_score(y_true_e12, y_pred_e12, average='macro'),
     CLR['bert']),
]
models = [r[0] for r in results]
accs   = [r[1] for r in results]
f1s    = [r[2] for r in results]
colors = [r[3] for r in results]

fig, ax1 = plt.subplots(figsize=(10, 5.5))
x  = np.arange(len(models))
w  = 0.5

bars = ax1.bar(x, accs, w, color=colors, edgecolor='white', linewidth=0.8, zorder=3)
for bar, acc_val, clr in zip(bars, accs, colors):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
             f'{acc_val:.2f}%', ha='center', va='bottom',
             fontsize=10, fontweight='bold', color=clr)

ax1.axhline(accs[0], color='#7F8C8D', linestyle='--', linewidth=1.2,
            alpha=0.8, label=f'Baseline E-01 ({accs[0]:.2f}%)')
ax1.set_ylabel('Akurasi (%)', fontweight='bold')
ax1.set_ylim(55, 102)
ax1.set_xticks(x); ax1.set_xticklabels(models, fontsize=9.5)
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.spines['top'].set_visible(False)

ax2 = ax1.twinx()
ax2.plot(x, f1s, 'o-', color=CLR['line'], linewidth=2.2,
         markersize=9, markerfacecolor='white',
         markeredgewidth=2.2, label='Macro F1-Score', zorder=4)
for xi, fi in zip(x, f1s):
    ax2.text(xi, fi+0.013, f'{fi:.2f}', ha='center', va='bottom',
             fontsize=9.5, fontweight='bold', color=CLR['line'])
ax2.set_ylabel('Macro F1-Score', fontweight='bold', color=CLR['line'])
ax2.set_ylim(0.3, 1.05)
ax2.tick_params(axis='y', labelcolor=CLR['line'])
ax2.spines['right'].set_color(CLR['line'])
ax2.spines['top'].set_visible(False)

ax1.set_title('Perbandingan Akurasi dan Macro F1-Score\nKeempat Eksperimen Utama', pad=12)

# Legend gabungan
p1 = mpatches.Patch(color=CLR['lr'],  label='Logistic Regression')
p2 = mpatches.Patch(color=CLR['xgb'], label='XGBoost Tuned v3.0')
p3 = mpatches.Patch(color=CLR['bert'],label='IndoBERT v2')
ln = plt.Line2D([0],[0],color=CLR['line'],marker='o',linewidth=2,
                markerfacecolor='white',markeredgewidth=2,label='Macro F1')
h1,l1 = ax1.get_legend_handles_labels()
ax1.legend(handles=[p1,p2,p3,ln]+h1, fontsize=8.5, ncol=2,
           loc='lower right', framealpha=0.9)

fig.tight_layout()
save_fig(fig, 'Gambar17_Perbandingan_Akurasi_F1.png')
plt.show()

## Gambar 18 — F1-Score Per Kelas Keempat Eksperimen

In [ ]:
from sklearn.metrics import classification_report

# Hitung F1 per kelas dari prediksi aktual
def get_f1_per_class(y_true, y_pred):
    rpt = classification_report(y_true, y_pred,
                                labels=['keluhan','saran','pujian'],
                                output_dict=True, zero_division=0)
    return (rpt['keluhan']['f1-score'],
            rpt['saran']['f1-score'],
            rpt['pujian']['f1-score'])

f1_e01 = get_f1_per_class(y_test_e01, y_pred_e01)
f1_e06 = get_f1_per_class(y_test_str,  y_pred_e06)
f1_e07 = get_f1_per_class(y_test_str,  y_pred_e07)
f1_e12 = get_f1_per_class(y_true_e12,  y_pred_e12)

model_labels = ['E-01\nLR Baseline', 'E-06\nLR Data Besar',
                'E-07\nXGBoost', 'E-12\nIndoBERT v2']
kel_f1 = [f1_e01[0], f1_e06[0], f1_e07[0], f1_e12[0]]
sar_f1 = [f1_e01[1], f1_e06[1], f1_e07[1], f1_e12[1]]
puj_f1 = [f1_e01[2], f1_e06[2], f1_e07[2], f1_e12[2]]

x  = np.arange(len(model_labels))
w  = 0.25

fig, ax = plt.subplots(figsize=(10, 5.5))
b1 = ax.bar(x-w,   kel_f1, w, label='Keluhan', color=CLR['keluhan'], edgecolor='white')
b2 = ax.bar(x,     sar_f1, w, label='Saran',   color=CLR['saran'],   edgecolor='white')
b3 = ax.bar(x+w,   puj_f1, w, label='Pujian',  color=CLR['pujian'],  edgecolor='white')

for group in [b1, b2, b3]:
    for bar in group:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.005,
                f'{h:.2f}', ha='center', va='bottom',
                fontsize=8, fontweight='bold')

ax.axhline(1.0, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
ax.set_title('Perbandingan F1-Score Per Kelas\nKeempat Eksperimen Utama', pad=12)
ax.set_ylabel('F1-Score', fontweight='bold')
ax.set_ylim(0, 1.12)
ax.set_xticks(x); ax.set_xticklabels(model_labels, fontsize=9.5)
ax.legend(fontsize=10, framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.1f}'))
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
save_fig(fig, 'Gambar18_F1_Per_Kelas.png')
plt.show()

print('\n✅ Semua gambar Bab 3 selesai!')
print(f'📁 Lokasi: {os.path.abspath(OUT)}')
for f in sorted(os.listdir(OUT)): print(f'   {f}')